# Evaluation of Experiments

In [15]:
import os
import json

import numpy as np
import pylab as plt
import pandas as pd
import yaml

from tqdm.auto import tqdm
from pathlib import Path
from omegaconf import OmegaConf

In [16]:
model_names = ['model_deterministic', 'model_ensemble', 'model_labelsmoothing', 'model_mcdropout', 'model_mixup', 'model_sngp']
A = {'model_deterministic' : 'DET', 'model_ensemble' : 'ENS', 'model_labelsmoothing' : 'LS', 'model_mcdropout' : 'MCD', 'model_mixup' : 'MIX', 'model_sngp' : 'SNGP'}
B = {'accuracy' : 'ACC', 'nll' : 'NLL', 'ace' : 'ACE', 'tce' : 'TCE', 'brier':'Brier', 'aupr_SVHN': 'AUPR (SVHN)', 'auroc_SVHN': 'AUROC (SVHN)', 'aupr_CIFAR100': 'AUPR (CIFAR100)', 'auroc_CIFAR100': 'AUROC (CIFAR100)'}
seeds = ['seed_1', 'seed_2', 'seed_3']

results = {mn:{} for mn in model_names}
root_path = Path('/home/phahn/repositories/dal-toolbox/examples/uncertainty/server_examples/server_results')

count = 0

for mn in model_names:
    paths = sorted(list(root_path.glob(mn+"/*")))
    for path in paths:
        if os.path.exists(path / 'results_final.json'):
            count += 1
            with open(path / 'results_final.json', 'r') as f:
                run_results = json.load(f)
            with open(path / '.hydra/config.yaml') as g:
                conf = yaml.load(g, Loader=yaml.FullLoader)

            results[mn][path.stem[-1]] = {'results': run_results, 'args': conf}
print(f"Number Runs: {count}/18.")

Number Runs: 18/18.


In [17]:
average_test_stats = {mn:{} for mn in model_names}

for mn, res in results.items():
    metrics = {}
    for seed, r in res.items():
        for met, val in (r['results']['id_test_stats'] | r['results']['ood_test_stats']).items():
            if met not in metrics:
                metrics[met] = [val]
            else:
                metrics[met].append(val)

    avg_metrics = {}
    for met, vals in metrics.items():
        avg_metrics[met] = np.mean(vals).round(4)
    
    average_test_stats[mn] = avg_metrics

In [23]:
dataframe = {mn:{} for mn in A.values()}
for mn, mets in average_test_stats.items():
    for met, val in mets.items():
        dataframe[A[mn]][B[met]] = val

pd.DataFrame(data=dataframe).T


,ACC,NLL,Brier,TCE,ACE,AUPR (SVHN),AUROC (SVHN),AUPR (CIFAR100),AUROC (CIFAR100)
DET,0.9518,0.1875,0.0773,0.0281,0.0340,0.9642,0.9319,0.8665,0.8797
ENS,0.9607,0.1309,0.0605,0.0087,0.0170,0.9779,0.9582,0.8970,0.9086
LS,0.9526,0.2021,0.0752,0.0424,0.0355,0.9360,0.8488,0.8411,0.8131
MCD,0.9445,0.1698,0.0824,0.0089,0.0153,0.9557,0.9202,0.8812,0.8978
MIX,0.9557,0.2081,0.0746,0.0659,0.0292,0.9620,0.9151,0.8515,0.8358
SNGP,0.9458,0.1769,0.0819,0.0057,0.0161,0.9779,0.9550,0.8723,0.8826
